In [37]:
# region Imports
import os
import sys
import torch
import numpy as np
from clustpy.deep.neural_networks.feedforward_autoencoder import FeedforwardAutoencoder
from sklearn.metrics import adjusted_mutual_info_score as ami
from sklearn.metrics import adjusted_rand_score as ari
from tqdm.notebook import tqdm
sys.path.append("/export/share/peters57dm/Verbund/deepsync/experiments/")
from helper.tracker import (
    AttractionRepellingLossTracker,
    AESyncLossTracker,
    EvaluationTracker,
)
from helper.datasets import (
    load_example,
    load_usps,
    load_htru,
    load_pendigits,
    load_optdigits,
    load_mnist,
    load_letterrecognition,
    load_cmu_faces,
    load_coil20,
    load_coil100,
    load_har,
    load_mice,
    load_synth_high,
    load_synth_low,
    load_weizmann,
    load_gaussian_blobs,
    load_cifar10,
    load_cifar100,
    load_fmnist,
    load_data,
)

sys.path.append("/export/share/peters57dm/Verbund/deepsync/")
from DeepSynC.helper import (
    # label assignment
    knn_assign_unlabeled_points,
    mahalanobis_assign_unlabeled_points,
    euclidean_assign_unlabeled_points,
    knn_average_assign_unlabeled_points,
    vote_of_two_knn_methods,
    # loss functions
    attraction_repelling_loss,
    ae_sync_loss,
    # run deep sync
    deep_sync_model_ship_trueK,
    deep_sync_model_ship
)
from helper.deep import (
    detect_device,
    load_pretrained_model,
    get_train_and_testloader,
    encode_batchwise
)
from helper.utils import save_dict_as_json, load_json_as_dict

In [38]:
# region Exp. Definition
experiment_params = {
    "loss_funs": [
        ("ae_sync_loss", ae_sync_loss),
        # ("att_rep_loss", attraction_repelling_loss)
    ],  # number of loss functions must be equal to # number of losses trackers
    "batch_loss_trackers": [
        AESyncLossTracker,
        # AttractionRepellingLossTracker
    ],
    "evaluation_tracker": EvaluationTracker,
    "datasets": [
        load_example,
        load_usps,
        load_htru,
        load_pendigits,
        load_optdigits,
        load_letterrecognition,
        load_har,
        load_mice,
        load_mnist,
        load_fmnist,
        load_coil20,
        load_coil100,
        load_weizmann,
        # load_synth_high,
        # load_synth_low,
        # load_cifar100,
        # load_cifar10,
        # load_gaussian_blobs,
    ],
    "label_assignment_methods": [
        ("knn_label_assignment", knn_assign_unlabeled_points),
        # ("mahalanobis_label_assignment", mahalanobis_assign_unlabeled_points),
        # ("knn_average_dist_label_assignment", knn_average_assign_unlabeled_points),
        # ("vote_knn_methods", vote_of_two_knn_methods)
        # ("euclidean_label_assignment", euclidean_assign_unlabeled_points)
    ],
    "experiment_repetitions": 1,
}
assert len(experiment_params["loss_funs"]) == len(experiment_params["batch_loss_trackers"])
# Hey! that is important too. Don't go too fast my friend :)
execution_params = {
    "experiment_root_path": "/export/share/peters57dm/Verbund/deepsync/results/experiments/DeepSync1NN",
    "model_name": "autoencoder.pth",
    "k": 25,
    "percent": 0.1,
    "n_check": 3,  # check for high confidence and check for early stopping.
    "learning_rate_pretrain": None,
    "learning_rate_deepsync": 1e-4,
    "pretrain_training_iterations": None,
    "clustering_training_iterations": 300,  # Switched to a max iteration of 300.
    "batch_size": 256,
    "max_embedded_dim_size": 10,
    "do_pretrain": False,
    "pretrain_loss": torch.nn.MSELoss(),
    "do_deepsync_train": True,
    "gif_duration": 300,
    # "sync_model_path": "/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison406/ae_sync_loss/knn_label_assignment",
    "sync_model_path_1": "/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison404/ae_sync_loss/knn_label_assignment",
    "sync_model_path_2": "/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison405/ae_sync_loss/knn_label_assignment",
    "AE_layers": [256, 128, 64],
    "Note": "In this experiment, we use SHiP with ground truth number of labels",
    "Note2": "Starting from code 400 and above, we used the new case of labeling assignment methods where only high confidence points can say something",
    "Note3": "removing the unified label method, 406: we use the best combination of deepsync and run for 5 pretrained models and let SHiP predict true K",
}
# endregion


# region Params Loading
base_path = execution_params["experiment_root_path"]
loss_fns = experiment_params["loss_funs"]
datasets_loading_methods = experiment_params["datasets"]
loss_trackers = experiment_params["batch_loss_trackers"]
labels_assignment_methods = experiment_params["label_assignment_methods"]
k = execution_params["k"]
percent = execution_params["percent"]
n_check = execution_params["n_check"]
pretrain_lr = execution_params["learning_rate_pretrain"]
deepsync_lr = execution_params["learning_rate_deepsync"]
pretrain_n_epochs = execution_params["pretrain_training_iterations"]
clustering_n_epochs = execution_params["clustering_training_iterations"]
batch_size = execution_params["batch_size"]
max_embed_size = execution_params["max_embedded_dim_size"]
do_pretrain = execution_params["do_pretrain"]
do_deepsync_train = execution_params["do_deepsync_train"]
pretrain_loss_fn = execution_params["pretrain_loss"]
experiment_repetitions = experiment_params["experiment_repetitions"]
# device = "cuda:1"
device = detect_device()
model_name = execution_params["model_name"]
sync_model_path_1 = execution_params["sync_model_path_1"]
sync_model_path_2 = execution_params["sync_model_path_2"]
deep_sync_model_name = "deep_sync_" + model_name
deep_sync_path = os.path.join(base_path, "deep_sync.pth")
ae_layers = execution_params["AE_layers"]
N_MODELS = 5
# endregion

In [ ]:
sys.path.append("/export/share/peters57dm/Verbund/deepsync/")
from helper.utils import save_dict_as_json, load_json_as_dict

# 406
def load_predicted_label_estimated_k(dataname, model_i):
    results_path = f"/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison406/ae_sync_loss/knn_label_assignment/{dataname}/model_0{model_i}/trackers/eval_tracker.json"
    results = load_json_as_dict(results_path)
    return np.array(results["predicted_labels"])

# 404 , 405
def load_predicted_label_true_k(dataname, model_i):
    results_path = f"/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison404/ae_sync_loss/knn_label_assignment/{dataname}/model_0{model_i}/trackers/eval_tracker.json"
    try :
        results_path = f"/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison404/ae_sync_loss/knn_label_assignment/{dataname}/model_0{model_i}/trackers/eval_tracker.json"
        results = load_json_as_dict(results_path)
    except FileNotFoundError:
        results_path = f"/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison405/ae_sync_loss/knn_label_assignment/{dataname}/model_0{model_i}/trackers/eval_tracker.json"
        results = load_json_as_dict(results_path)
    return np.array(results["predicted_labels"])

In [ ]:
# 406
# def load_results(dataname, model_i):
#     results_path = f"/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison406/ae_sync_loss/knn_label_assignment/{dataname}/model_0{model_i}/trackers/eval_tracker.json"
#     results = load_json_as_dict(results_path)
#     return results

# 404, 405
def load_results(dataname, model_i):
    try :
        results_path = f"/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison404/ae_sync_loss/knn_label_assignment/{dataname}/model_0{model_i}/trackers/eval_tracker.json"
        results = load_json_as_dict(results_path)
    except FileNotFoundError:
        results_path = f"/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison405/ae_sync_loss/knn_label_assignment/{dataname}/model_0{model_i}/trackers/eval_tracker.json"
        results = load_json_as_dict(results_path)
    return results

In [40]:
from sklearn.neighbors import KNeighborsClassifier as KNN

In [41]:
results = {}
for ds_loader in tqdm(datasets_loading_methods, total=len(datasets_loading_methods)):
    data, gt_labels, data_name = load_data(ds_loader)
    embedded_space_dim = min(data.shape[1], max_embed_size)
    for model_i in range(N_MODELS):
        
        model = FeedforwardAutoencoder(
            layers=[data.shape[1], ae_layers[0], ae_layers[1], ae_layers[2], embedded_space_dim]
        ).to(device)
        try :
            _mpath = os.path.join(sync_model_path_1, data_name, f"model_0{model_i}", "deep_sync_autoencoder.pth")
            model = load_pretrained_model(model, _mpath, device)
        except FileNotFoundError:
            _mpath = os.path.join(sync_model_path_2, data_name, f"model_0{model_i}", "deep_sync_autoencoder.pth")
            model = load_pretrained_model(model, _mpath, device)
        _, testloader = get_train_and_testloader(data, gt_labels, batch_size)
        embedded, gt_labels = encode_batchwise(testloader, model, device)
        np_gt_labels = gt_labels.numpy()

        predicted_labels = load_predicted_label(data_name, model_i)
        labeled_points_mask = predicted_labels > -1
        labeled_points = embedded[labeled_points_mask]
        corresponding_labels = np_gt_labels[labeled_points_mask]
        if len(embedded[~labeled_points_mask]) == 0:
            if not data_name in results:
                results[data_name] = {}
            results[data_name][f"model_0{model_i}"] = "All points are alread labeled."
            continue
        knn = KNN(1).fit(labeled_points, corresponding_labels)
        knn_predictions = knn.predict(embedded[~labeled_points_mask])
        nn_predictions = np.copy(np_gt_labels)
        nn_predictions[~labeled_points_mask] = knn_predictions
        if not data_name in results:
            results[data_name] = {}
        results[data_name][f"model_0{model_i}"] = {}
        results[data_name][f"model_0{model_i}"]["NN_predictions"] = nn_predictions.tolist()
        results[data_name][f"model_0{model_i}"]["original_predictions"] = predicted_labels.tolist()
        results[data_name][f"model_0{model_i}"]["gt_labels"] = np_gt_labels.tolist()
        results[data_name][f"model_0{model_i}"]["ami"] = ami(nn_predictions, predicted_labels)
        results[data_name][f"model_0{model_i}"]["ari"] = ari(nn_predictions, predicted_labels)
save_dict_as_json(
    results, os.path.join(base_path, f"comparison40_4-5_models_results.json")
)

  0%|          | 0/13 [00:00<?, ?it/s]

/export/share/peters57dm/Verbund/deepsync/experiments/helper/deep.py:108: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(pretrained_model_path, map_lo

IndexError: boolean index did not match indexed array along dimension 0; dimension is 6108 but corresponding boolean dimension is 5701

In [51]:
__p = load_predicted_label(data_name, 0)
__p.shape

(5701,)

In [49]:
predicted_labels.shape

(5701,)

In [42]:
results.keys()

dict_keys(['example', 'USPS', 'htru', 'pendigits', 'optdigits', 'letterrecognition', 'HAR', 'mice', 'MNIST', 'FMNIST', 'coil20', 'coil100'])

In [43]:
avg_ari_ami = {}
for d in results:
    models = results[d]
    _aris = []
    _amis = []
    _ks = []
    _n_lbld = []
    avg_ari_ami[d] = {}
    for m in models:
        predictions = models[m]
        if isinstance(predictions, str):
            print(predictions)
            print(d)
            print(m)
            rr = load_results(d, int(m.split('_')[-1]))
            _aris.append(rr['ari_labeled'][-1])
            _amis.append(rr['ami_labeled'][-1])
            _ks.append(len(np.unique(rr['predicted_labels'])))
            _n_lbld.append(len(rr['predicted_labels']))
            continue
        _aris.append(float(predictions['ari']))
        _amis.append(float(predictions['ami'])) 
        _ks.append(np.sum(np.unique(
            np.array(predictions['original_predictions'])) > -1))
        _n_lbld.append(np.sum(
            np.array(predictions['original_predictions']) > -1))

    avg_ari_ami[d]['ari'] = f"{np.mean(_aris):.4f}±{np.std(_aris):.4f}"
    avg_ari_ami[d]['ami'] = f"{np.mean(_amis):.4f}±{np.std(_amis):.4f}"
    avg_ari_ami[d]['predicted_k'] = f"{np.mean(_ks):.4f}±{np.std(_ks):.4f}"
    avg_ari_ami[d]['n_labeled_points'] = f"{np.mean(_n_lbld):.4f}±{np.std(_n_lbld):.4f}" 

All points are alread labeled.
HAR
model_01
All points are alread labeled.
HAR
model_02
All points are alread labeled.
HAR
model_04


In [44]:
from pprint import pprint
pprint(avg_ari_ami)

{'FMNIST': {'ami': '0.6069±0.0251',
            'ari': '0.3909±0.0249',
            'n_labeled_points': '68370.0000±529.4582',
            'predicted_k': '10.0000±0.0000'},
 'HAR': {'ami': '0.6023±0.0259',
         'ari': '0.4532±0.0545',
         'n_labeled_points': '10290.4000±11.8254',
         'predicted_k': '10.2000±3.9192'},
 'MNIST': {'ami': '0.8154±0.0207',
           'ari': '0.7432±0.0487',
           'n_labeled_points': '69314.2000±86.0172',
           'predicted_k': '10.0000±0.0000'},
 'USPS': {'ami': '0.7566±0.0133',
          'ari': '0.7152±0.0211',
          'n_labeled_points': '8641.4000±211.2227',
          'predicted_k': '10.0000±0.0000'},
 'coil100': {'ami': '0.5733±0.0170',
             'ari': '0.0414±0.0046',
             'n_labeled_points': '3707.2000±155.6540',
             'predicted_k': '100.0000±0.0000'},
 'coil20': {'ami': '0.6003±0.0229',
            'ari': '0.2007±0.0333',
            'n_labeled_points': '826.4000±58.0572',
            'predicted_k': '20.000

In [45]:
save_dict_as_json(
    avg_ari_ami, "/export/share/peters57dm/Verbund/deepsync/results/experiments/DeepSync1NN/avg_results_trueK.json"
)

In [46]:
import pandas as pd
pd.DataFrame(avg_ari_ami).to_excel("/export/share/peters57dm/Verbund/deepsync/results/experiments/DeepSync1NN/avg_results_trueK.xlsx")